## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import random
import json
import re
import math
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestNeighbors

from haystack import Pipeline, component
from hayhooks import BasePipelineWrapper
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
from sem_rag.config.settings import VLLM, Qdrant, Evaluation, SEMDataset
from sem_rag.retrieval.bm25 import BM25Retriever
from sem_rag.embedding.embedder import CLIPEmbedder
from sem_rag.image.encoding import encode_image
from sem_rag.storage.qdrant import get_document_store
from sem_rag.image.descriptions import get_crop_path

In [ ]:
INSTRUCTION = "Represent the user's input."

TOP_K = 5

In [ ]:
QUERIES = [
    "SEM image showing spherical nanoparticles",
]

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

In [ ]:
qwen_qdrant = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)
clip_qdrant = get_document_store(Qdrant.COLLECTION_CLIP, Qdrant.EMBEDDING_DIM_CLIP)

# First Evaluation Method: LLM-as-Judge

## BM25

In [ ]:
bm25_retriever = BM25Retriever(Evaluation.BM25_DIR)

In [ ]:
bm25_results = bm25_retriever.search(
    QUERIES[0],
    top_k=TOP_K,
)

for result in bm25_results:
    print(result["image_id"], result["score"], result["description"])
    print("_"*100)

## CLIP

In [ ]:
clip_embedder = CLIPEmbedder()
retriever = QdrantEmbeddingRetriever(clip_qdrant)

In [ ]:
# Generate the CLIP text embedding
query_embedding = clip_embedder.embed_text(QUERIES[0])

In [ ]:
# Retrieve the most similar images
clip_results = retriever.run(
    query_embedding=query_embedding,
    top_k=TOP_K,
).get("documents", [])

In [ ]:
for result in clip_results:
    print(result)

## RAG System

In [ ]:
@component
class QueryEmbedder():
    def __init__(self, model: str, api_base_url: str, api_key: str):
        self.model = model
        self.client = OpenAI(base_url=api_base_url, api_key=api_key)

    @component.output_types(embedding=list[float])
    def run(self, text: str = "", image_data_urls: list[str] | None = None):
        if image_data_urls:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": image_data_urls[0]}},
                        {"type": "text", "text": ""},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]
        else:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [{"type": "text", "text": text}]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]

        response: CreateEmbeddingResponse = self.client.post(
            "/embeddings",
            cast_to=CreateEmbeddingResponse,
            body={
                "messages": messages,
                "model": self.model,
                "encoding_format": "float",
                "continue_final_message": True,
                "add_special_tokens": True,
            },
        )

        return {"embedding": response.data[0].embedding}

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)

        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder(model=VLLM.EMBEDDING_MODEL, api_base_url=VLLM.API_URL, api_key=VLLM.API_KEY))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=TOP_K))

        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")

    def retrieve_documents(self, question: str, images: list[str]):
        retrieval_run_args = {
            "query_embedder": {
                "text": question,
                "image_data_urls": images,
            },
        }

        response = self.retrieval_pipeline.run(retrieval_run_args)

        documents = (response.get("retriever", {}).get("documents", []))

        return documents

In [ ]:
qwen_retriever = PipelineWrapper()
qwen_retriever.setup()

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    QUERIES[0],
    [],
)

for result in qwen_results:
    print(result.id, result.content)

## LLM as a Judge

### Qwen3.5 122B A10B

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of image retrieval systems for
Scanning Electron Microscopy (SEM) images.

Your task is to judge how relevant a candidate SEM image is
to a user's retrieval query.

The candidate IMAGE is the PRIMARY source of evidence for your judgment.

The candidate DESCRIPTION is SECONDARY supporting information.
Use it only to clarify or supplement what is visible in the image.

If the description conflicts with what is visible in the image,
trust the image.

Do not consider information present only in the description as
evidence unless it can reasonably be supported by the image.

Evaluate the candidate independently from other candidates.

Consider:
- morphology
- particle shape
- particle size when reliably inferable
- structure
- texture
- spatial arrangement
- visible patterns
- other visual characteristics explicitly requested by the query

Do not reward an image merely because it is an SEM image.

Do not invent information that is not visible in the image.

If the query specifies a numerical property such as particle size,
do not assume that value from visual appearance alone unless the
image provides sufficient evidence, such as a scale bar.

Use this relevance scale:

0 = Not relevant
1 = Slightly relevant
2 = Moderately relevant
3 = Highly relevant
4 = Extremely relevant / excellent match

Return ONLY valid JSON in exactly this format:

{
  "relevance": <integer from 0 to 4>
}

Do not include markdown.
Do not include explanations.
Do not include any additional fields.
"""

In [ ]:
def judge_candidate(
    query: str,
    image_url: str,
    description: str | None = None,
) -> dict:

    user_content = [
        {
            "type": "text",
            "text": f"""
            Query:
            {query}

            The following image is the candidate being evaluated.
            The IMAGE is the primary evidence for your judgment.
            """,
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            },
        },
    ]

    if description:
        user_content.append(
            {
                "type": "text",
                "text": f"""
                Supporting candidate description:
                {description}

                Use this description only as secondary supporting information.
                Base your relevance judgment primarily on the image.
                """,
            }
        )

    response = client.chat.completions.create(
        model=VLLM.JUDGE_MODEL, 
        messages=[
            {
                "role": "system",
                "content": JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_content,
            },
        ],
        temperature=0.0,
    )

    raw_output = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw_output)

    except json.JSONDecodeError:
        match = re.search(
            r"\{.*\}",
            raw_output,
            re.DOTALL,
        )

        if not match:
            raise ValueError(
                f"Could not parse judge response:\n{raw_output}"
            )

        result = json.loads(match.group(0))

    relevance = result.get("relevance")

    if relevance not in [0, 1, 2, 3, 4]:
        raise ValueError(
            f"Invalid relevance score: {relevance}\n"
            f"Raw response: {raw_output}"
        )

    return {
        "relevance": relevance,
        # "reason": result.get("reason", ""),
        # "raw_response": raw_output,
    }

In [ ]:
image_url = encode_image(SEMDataset.CROPS / "38_a6a840c41ec3a131a5d6de121831afb7797e181926bfdfed50b62b3335899f52_512.npy")

query = "SEM image showing spherical particles around 5 micrometers"

result = judge_candidate(
    query=query,
    image_url=image_url,
    description=None,
)

print(result)

## Results

In [ ]:
def evaluate_queries(query, bm25_results, clip_results, qwen_results):
    evaluation = []

    # 1. Get the rankings from both systems
    bm25_ranking = [result["image_id"] for result in bm25_results]
    clip_ranking = [result.id for result in clip_results]
    qwen_ranking = [result.id for result in qwen_results]

    # 2. Create the union of the two rankings (set to remove duplicates)
    pool = set(bm25_ranking) | set(qwen_ranking) | set(clip_ranking)

    # 3. Judge every unique image only once
    relevance = {}

    for image_id in pool:
        doc = qwen_qdrant.get_documents_by_id([image_id])[0]

        image_thumb = doc.meta.get("thumb_b64")
        image_description = doc.content

        score = judge_candidate(
            query,
            image_thumb,
            image_description,
        )

        relevance[image_id] = score["relevance"]

    # 4. Store everything together
    evaluation.append({
        "query": query,
        "bm25": bm25_ranking,
        "clip": clip_ranking,
        "qwen": qwen_ranking,
        "relevance": relevance,
    })

    return evaluation

In [ ]:
evaluation = evaluate_queries(
    QUERIES[0],
    bm25_results,
    clip_results,
    qwen_results,
)

evaluation

In [ ]:
def dcg(relevances):
    return sum(
        (2 ** rel - 1) / math.log2(i + 2)
        for i, rel in enumerate(relevances)
    )


def ndcg_at_k(system_ranking, relevance, k=5):
    # Relevance scores in the order returned by the system
    system_relevances = [
        relevance[image_id]
        for image_id in system_ranking[:k]
    ]

    # Ideal ranking comes from the entire common pool
    ideal_relevances = sorted(
        relevance.values(),
        reverse=True
    )[:k]

    ideal_dcg = dcg(ideal_relevances)

    if ideal_dcg == 0:
        return 0.0

    return dcg(system_relevances) / ideal_dcg

In [ ]:
bm25_ndcgs = []
clip_ndcgs = []
qwen_ndcgs = []

for result in evaluation:

    bm25_score = ndcg_at_k(
        result["bm25"],
        result["relevance"],
        k=TOP_K,
    )

    clip_score = ndcg_at_k(
        result["clip"],
        result["relevance"],
        k=TOP_K,
    )
    
    qwen_score = ndcg_at_k(
        result["qwen"],
        result["relevance"],
        k=TOP_K,
    )

    bm25_ndcgs.append(bm25_score)
    clip_ndcgs.append(clip_score)
    qwen_ndcgs.append(qwen_score)

In [ ]:
print("BM25:", sum(bm25_ndcgs) / len(bm25_ndcgs))
print("CLIP:", sum(clip_ndcgs) / len(clip_ndcgs))
print("InternRAG:", sum(qwen_ndcgs) / len(qwen_ndcgs))

```bash
BM25: 0.870429202289663
CLIP: 0.5018852674630611
InternRAG: 0.9299155484975696
```

# Second Evaluation Method: Target-Retrieval / Image Identification Task

A manually constructed query that provides sufficient visual information to uniquely identify the target image within the evaluation dataset.

```text
This also gives you a much cleaner experimental distinction:
Text retrieval
Can a system map a detailed textual description of an SEM image to the corresponding image?
versus
Image retrieval
Can a system retrieve visually/morphologically similar images given an SEM image?
versus
Metadata retrieval
Can a system retrieve images satisfying structured acquisition/property constraints?
```

In [ ]:
files = sorted(SEMDataset.EMBEDDINGS_DINO.glob("*.npz"))

In [ ]:
data = np.load(files[0])

In [ ]:
data

In [ ]:
print(data["sample_id"])
print(data["cls"])
print(data["meanpatch"])

In [ ]:
data["meanpatch"]

In [ ]:
embeddings = []
image_ids = []

for f in files:
    data = np.load(f)

    embeddings.append(data["cls"])

    # ID = everything before the first "_"
    image_id = f.name.split("_")[0]
    image_ids.append(image_id)

embeddings = np.vstack(embeddings)

print(f"Loaded {len(embeddings)} embeddings")
print(f"Shape: {embeddings.shape}")
print(f"First image ID: {image_ids[0]}")

Find the 5 nearest neighbours using cosine distance (of course the first image will be the image itself).

In [ ]:
nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)
nn.fit(embeddings)

distances, indices = nn.kneighbors(embeddings)

# The first neighbour is the image itself
mean_5nn_distance = distances[:, 1:].mean(axis=1)

In [ ]:
low_threshold = np.quantile(mean_5nn_distance, 1/3)
high_threshold = np.quantile(mean_5nn_distance, 2/3)
# highest_threshold = np.quantile(mean_5nn_distance, 99/100)

low = np.where(mean_5nn_distance <= low_threshold)[0]
medium = np.where(
    (mean_5nn_distance > low_threshold) &
    (mean_5nn_distance <= high_threshold)
)[0]
high = np.where(
    (mean_5nn_distance > high_threshold) #&
    # (mean_5nn_distance <= highest_threshold)
)[0]
# highest = np.where((mean_5nn_distance))

print(f"Low:     {len(low)}")
print(f"Medium:  {len(medium)}")
print(f"High:    {len(high)}")
# print(f"Highest: {len(highest)}")

In [ ]:
rng = np.random.default_rng(0)

selected = {
    "low": rng.choice(low),
    "medium": rng.choice(medium),
    "high": rng.choice(high),
    # "highest": rng.choice(highest)
}

for group, idx in selected.items():
    print(
        f"{group:7s} | "
        f"sample_id={image_ids[idx]} | "
        f"mean 5-NN distance={mean_5nn_distance[idx]:.4f}"
    )

In [ ]:
idx = np.argmax(mean_5nn_distance)

In [ ]:
image_ids[idx]

#### The following are just some poor tries

In [ ]:
random.seed(0)

In [ ]:
crop_files = list(SEMDataset.CROPS.glob("*.npy"))

crop_path = random.choice(crop_files)

In [ ]:
crop_path = get_crop_path(25323)

In [ ]:
image = np.load(crop_path)

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title(crop_path.name)
plt.axis("off")
plt.show()

print(f"Selected image: {crop_path}")
print(f"Shape: {image.shape}")

In [ ]:
natural_query = "Find the SEM image of square-topped micro-pillars with ridged, textured sidewalls, arranged in a grid array, viewed at an angle with visible shadows"

In [ ]:
natural_query = "SEM image of an irregular branching light-gray flake or deposit with finger-like extensions on a dark background"

In [ ]:
image = "Find the SEM image showing an array of free-standing, vertically-oriented micro-pillars (posts) fabricated on a flat substrate, imaged at an oblique/tilted viewing angle (not top-down)." \
"Each pillar has a flat, smooth, square (or near-square) top facet with slightly rounded corners. The top surfaces appear relatively smooth and featureless compared to the sidewalls. The pillars taper very slightly — marginally narrower partway down before widening again near the base — giving a subtly hourglass/waisted silhouette rather than a perfectly straight column." \
"The vertical sidewalls of each pillar show a pronounced horizontal striated/ridged texture — fine, closely-spaced parallel bands running perpendicular to the pillar axis (circumferential striations), consistent with a layer-by-layer or step-wise etching/growth process. This ridged texture is the most visually distinctive feature of the image." \
"1 pillar is completely visible, while there 5 pillars that are only partially visible. They are arranged in a regular grid pattern (rows/columns), consistent with a lithographically patterned array. Pillars are cut off by the image frame on multiple edges, indicating the array extends beyond the field of view." \
"Each pillar casts a dark, sharply-defined trapezoidal/parallelogram-shaped shadow on the substrate below and to one side, all shadows oriented in the same direction — indicating a single, fixed illumination/detector geometry (consistent with SEM secondary-electron imaging, not optical shadowing). The substrate itself appears as a uniform mid-gray, slightly textured flat plane." \
"Classic SEM grayscale contrast — pillar tops and upper sidewalls appear bright/high-contrast (edge/charging effects), while shadowed regions and the substrate are darker. No color, no scale bar visible in the cropped region, no annotations overlaid on the image itself."

In [ ]:
image = "An isolated, irregular gray structure with a broad central body and several narrow elongated protrusions. Three prominent protrusions extend toward the left, two extend toward the lower-left, and one larger protrusion extends upward-right. The structure has a bright, sharply defined outline and is surrounded by a predominantly dark, featureless background."

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    "Find the SEM image of micro-pillars arranged in a grid array",
    [],
)

for result in qwen_results:
    # if result.id == "81328":
        print(result.id, result.content)

In [ ]:
image = np.load(get_crop_path(50697))

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title(get_crop_path(81333).name)
plt.axis("off")
plt.show()

print(f"Selected image: {get_crop_path(81333)}")
print(f"Shape: {image.shape}")

In [ ]:
image = np.load(get_crop_path(55402))

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title(get_crop_path(81315).name)
plt.axis("off")
plt.show()

print(f"Selected image: {get_crop_path(81315)}")
print(f"Shape: {image.shape}")

In [ ]:
image = np.load(get_crop_path("81353"))

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title(get_crop_path("81353").name)
plt.axis("off")
plt.show()

print(f"Selected image: {get_crop_path("81353")}")
print(f"Shape: {image.shape}")

In [ ]:
k = 5

nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine")
nn.fit(embeddings)

distances, indices = nn.kneighbors(embeddings)

mean_distance = distances[:, 1:].mean(axis=1)

idx = np.argmax(mean_distance)

print("Target:", image_ids[idx])
print("Mean distance:", mean_distance[idx])

for rank, (neighbour_idx, distance) in enumerate(
    zip(indices[idx, 1:], distances[idx, 1:]),
    start=1
):
    print(
        f"{rank}: {image_ids[neighbour_idx]} "
        f"(distance={distance:.4f})"
    )

In [ ]:
image = np.load(get_crop_path("85075"))

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title(get_crop_path("81353").name)
plt.axis("off")
plt.show()

print(f"Selected image: {get_crop_path("81353")}")
print(f"Shape: {image.shape}")